### Phase 4 (Modeling)

In [1]:
import os
from pathlib import Path
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor


In [2]:
DATA_PATH = Path(os.getenv("DATA_PATH"))  # oikea polku tähän

# ══════ Lue levyltä ══════
train_df = pl.read_parquet(DATA_PATH / "train.parquet")
test_df  = pl.read_parquet(DATA_PATH / "test.parquet")

# ══════ X/y-jako ══════
TARGETS = ["price", "daysonmarket"]

X_train = train_df.drop(TARGETS)
X_test  = test_df.drop(TARGETS)

y_price_train = train_df["price"]
y_price_test  = test_df["price"]
y_dom_train   = train_df["daysonmarket"]
y_dom_test    = test_df["daysonmarket"]

### Ladataan data

In [3]:

from pathlib import Path
import polars as pl


def load_dataset(model_type: str, target: str, data_path: Path):
    """
    model_type: 'encoded' | 'catboost'
    target: 'price' | 'daysonmarket'
    """

    base = data_path / model_type

    X_train = pl.read_parquet(base / "X_train.parquet").to_pandas()
    X_test  = pl.read_parquet(base / "X_test.parquet").to_pandas()

    y_train = (
        pl.read_parquet(base / f"y_{target}_train.parquet")
        .to_pandas()
        .squeeze()
    )

    y_test = (
        pl.read_parquet(base / f"y_{target}_test.parquet")
        .to_pandas()
        .squeeze()
    )

    date_cols = [col for col in X_train.columns if X_train[col].dtype == "datetime64[ms]"]
    print(date_cols)  # tarkista mitkä sarakkeet poistetaan

    X_train = X_train.drop(columns=date_cols)
    X_test  = X_test.drop(columns=date_cols)

    return X_train, X_test, y_train, y_test




In [4]:
# Catboostille ei hyödy label-enkoodauksesta -> tästä suoraan kolutus ja testi
# ja targetteja ei label-enkoodata koskaan
X_train_for_catboost = X_train.clone()
X_test_catboost  = X_test.clone()

print(X_train.dtypes)

date_cols = [col for col in X_train.columns if X_train[col].dtype == "datetime64[ms]"]
print(date_cols) 


#label-enkoodaus kategorille LightGBM ja XGBoostia varten

categorical_cols = [
    # Perus kategoriat
    "make_name",
    "model_name",
    "body_type",
    "engine_type",
    "fuel_type",
    "wheel_system",
    "trans_group",
    "listed_season",
    "listed_month",
    "listed_weekday", 
    "age_bin",
    "mileage_bin",
    "GEOID",
    "STATEFP",
]


def apply_label_encoders(df, encoders):
    df_out = df.clone()

    for col, mapping in encoders.items():
        df_out = df_out.with_columns(
            pl.col(col)
            .replace_strict(mapping, default=-1)
            .cast(pl.Int32)
            .alias(col)
        )

    return df_out



def fit_label_encoders(train_df, categorical_cols):
    encoders = {}

    for col in categorical_cols:
        categories = (
            train_df.select(col)
            .unique()
            .to_series()
            .to_list()
        )

        mapping = {cat: i for i, cat in enumerate(categories)}
        encoders[col] = mapping

    return encoders

encoders = fit_label_encoders(X_train, categorical_cols)

X_train_encoded = apply_label_encoders(X_train, encoders)
X_test_encoded  = apply_label_encoders(X_test, encoders)

def save_dataset(X_train, X_test, y_price_train, y_price_test,
                 y_dom_train, y_dom_test, data_path: Path, model_type: str):
    
    base = data_path / model_type
    base.mkdir(parents=True, exist_ok=True)

    X_train.write_parquet(base / "X_train.parquet")
    X_test.write_parquet(base / "X_test.parquet")

    y_price_train.to_frame().write_parquet(base / "y_price_train.parquet")
    y_price_test.to_frame().write_parquet(base / "y_price_test.parquet")
    y_dom_train.to_frame().write_parquet(base / "y_daysonmarket_train.parquet")
    y_dom_test.to_frame().write_parquet(base / "y_daysonmarket_test.parquet")

    print(f"Tallennettu: {base}")


save_dataset(
    X_train_encoded, X_test_encoded,
    y_price_train, y_price_test,
    y_dom_train, y_dom_test,
    DATA_PATH, model_type="encoded"
)

X_train_catboost = X_train.clone()
X_test_catboost  = X_test.clone()

save_dataset(
    X_train_catboost, X_test_catboost,
    y_price_train, y_price_test,
    y_dom_train, y_dom_test,
    DATA_PATH, model_type="catboost"
)

[String, Float64, String, Float64, String, Float64, Float64, Float64, String, String, String, Int64, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, Int8, String, String, Int8, Int8, Int32, Float64, Float64, Float64, Int64, String, String, UInt32, Int8, String, String, Float64, Float64, Float64, Float64, Float64, Float64, Float64]
[]
Tallennettu: <PROJECT_DIR>\cars\encoded
Tallennettu: <PROJECT_DIR>\cars\catboost


### Malli: XGBoost

In [5]:
TARGET = "daysonmarket"

X_train, X_test, y_train, y_test = load_dataset(
    model_type="encoded",
    target=TARGET,
    data_path=DATA_PATH
)

print(X_train.shape, X_test.shape)

xgb_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.06,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1
)

print(f"Training XGBoost {TARGET} ")
xgb_model.fit(X_train, y_train)
print("✅ Training done.")


# Predictions
xgb_pred = xgb_model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print("\n=== XGBoost tulokset ===")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")

[]
(31946, 59) (7558, 59)
Training XGBoost daysonmarket 
✅ Training done.

=== XGBoost tulokset ===
MAE : 79.24
RMSE: 90.08
R²  : -638.8481


### Malli: CatBoost

In [7]:
TARGET = 'daysonmarket'

X_train, X_test, y_train, y_test = load_dataset(
    model_type="catboost",
    target=TARGET,
    data_path=DATA_PATH
)

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

cat_model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    verbose=False
)

print("⏳ Training CatBoost...")

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

print("✅ Training done.")

# Predictions
cat_pred = cat_model.predict(X_test)


# Metrics
mae = mean_absolute_error(y_test, cat_pred)
rmse = np.sqrt(mean_squared_error(y_test, cat_pred))
r2 = r2_score(y_test, cat_pred)

medianAE = np.median(np.abs(y_test - cat_pred))

print("\n=== CatBoost tulokset ===")
print(f"MAE : {mae:,.2f}")
print(f"medianAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")




[]
⏳ Training CatBoost...


C:\Users\user\AppData\Local\Temp\ipykernel_3568\1253197443.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


✅ Training done.

=== CatBoost tulokset ===
MAE : 121.39
medianAE : 121.39
RMSE: 142.82
R²  : -1607.4085


### Malli: LightGBM

In [ ]:
TARGET = "price"

X_train, X_test, y_train, y_test = load_dataset(
    model_type="encoded",
    target=TARGET,
    data_path=DATA_PATH
)


lgb_model = lgb.LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

print("⏳ Training LightGBM...")
lgb_model.fit(X_train, y_train)
print("✅ Training done.")

# Predictions
lgb_pred = lgb_model.predict(X_test)

# Metrics
mae  = mean_absolute_error(y_test, lgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
medianAE = np.median(np.abs(y_test - lgb_pred))

r2   = r2_score(y_test, lgb_pred)

print("\n=== LightGBM tulokset ===")
print(f"MAE : {mae:,.2f}")
print(f"medianAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")

[]
⏳ Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004478 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3597
[LightGBM] [Info] Number of data points in the train set: 31946, number of used features: 55
[LightGBM] [Info] Start training from score 26263.130501
✅ Training done.

=== LightGBM tulokset ===
MAE : 2,328.29
medianAE : 2,328.29
RMSE: 4,219.25
R²  : 0.9136


### Malli: RandomForest

In [ ]:
# rf_model = RandomForestRegressor(
#     n_estimators=200,
#     max_depth=20,
#     n_jobs=-1
# )

# print("⏳ Training RandomForest...")
# rf_model.fit(X_train, y_train)
# print("✅ Training done.")

# # Predictions
# rf_pred = rf_model.predict(X_test)

# # Metrics
# mae  = mean_absolute_error(y_test, rf_pred)
# rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
# r2   = r2_score(y_test, rf_pred)

# print("\n=== RandomForest tulokset ===")